# Spot-Checking: Model Comparison

This notebook performs a detailed comparison of classification models using the **best configuration identified in previous spot-checking experiments**:

- **Balancing:** None (identified as the best strategy in `Spot_Checking_Balancing.ipynb`)
- **Dimensionality Reduction:** PCA - 90% variance (identified as the best DR technique in `Spot_Checking_DimensionalityReduction.ipynb`)

**Pipeline per fold:** `MinMaxScaler -> PCA(n_components=0.90) -> Classifier`  
All transformations are fitted only on the training data of each fold (no data leakage).

**Evaluation:**
- Stratified cross-validation (`StratifiedKFold`, k=5)
- Metrics: F1, Accuracy, Precision, Recall
- Visualizations: F1 boxplots, confusion matrices, ROC curves

## 1. Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score,precision_recall_fscore_support
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
)

# Classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

FIGURES_DIR = PROJECT_ROOT / "reports" / "paper" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from src.modeling_utils import make_modeling_pipeline, make_stratified_cv
from src.visualization_utils import save_dataframe_as_figure

import warnings
warnings.filterwarnings('ignore')

## 2. Load and Prepare Data

In [ ]:
df = pd.read_csv("../data/processed/kidney_disease_encoded.csv")

print(f"Shape: {df.shape}")
print(f"\nTarget variable distribution:")
print(df["class"].value_counts())

In [ ]:
X = df.drop(columns=["class"])
y = LabelEncoder().fit_transform(df["class"])  # ckd=0, notckd=1 (or vice versa)

print(f"Features: {X.shape[1]} columns, {X.shape[0]} samples")
print(f"Encoded classes: {np.unique(y)} (original: {df['class'].unique()})")

## 3. Pipeline

Pipeline locked to the best configuration found in spot-checking:
1. **`MinMaxScaler`:** fitted only on the training fold
2. **`PCA(n_components=0.90)`:** retains 90% of variance; fitted only on the training fold
3. **`Classifier`:** interchangeable

No balancing applied. Helper: `make_modeling_pipeline` from `src/modeling_utils`.

In [ ]:
def make_pipeline(classifier):
    return make_modeling_pipeline(
        classifier=classifier,
        reducer=PCA(n_components=0.9, random_state=42),
    )

## 4. Models Definition

> `SVC` is instantiated with `probability=True` to enable `predict_proba`, required for ROC curve computation via `cross_val_predict`.

In [ ]:
models = {
    "KNN":   KNeighborsClassifier(),
    "DTree": DecisionTreeClassifier(random_state=42),
    "RF":    RandomForestClassifier(random_state=42),
    "SVM":   SVC(probability=True, random_state=42),  # probability=True required for ROC curves
    "NB":    GaussianNB(),
}

## 5. Cross-Validation - Performance Metrics

In [ ]:
cv      = make_stratified_cv()
scoring = ['accuracy', 'precision', 'recall', 'f1']

all_scores = {}

for model_name, model_candidate in models.items():
    pipeline = make_pipeline(model_candidate)
    scores   = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=False)
    all_scores[model_name] = scores
    print(f"Done: {model_name}")

print("\nAll done.")

## 6. Summary Table

In [ ]:
rows = []
for model_name, scores in all_scores.items():
    row = {"Model": model_name}
    for metric in scoring:
        row[f"{metric}_mean"] = np.mean(scores[f"test_{metric}"])
        row[f"{metric}_std"]  = np.std(scores[f"test_{metric}"])
    rows.append(row)

summary_df = (
    pd.DataFrame(rows)
    .sort_values("f1_mean", ascending=False)
    .set_index("Model")
)

float_cols = summary_df.select_dtypes(include="float").columns.tolist()
mean_cols  = [c for c in float_cols if c.endswith("_mean")]

display(
    summary_df.style
    .format("{:.4f}", subset=float_cols)
    .background_gradient(cmap="RdYlGn", axis=0, subset=mean_cols)
)

## 7. F1 Boxplot - Distribution per Model (CV Folds)

In [ ]:
f1_data = {
    model_name: scores["test_f1"].tolist()
    for model_name, scores in all_scores.items()
}

fig, ax = plt.subplots(figsize=(10, 5))
pd.DataFrame(f1_data).boxplot(ax=ax)
ax.set_title("F1-score Distribution per Model\n(PCA 90% var, No Balancing, k=5 CV)")
ax.set_ylabel("F1-score")
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "spot_checking_model_comparison_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

## No DR


### 7.b Summary Table - No DR

In [ ]:
all_scores_no_dr = {}

for model_name, model_candidate in models.items():
    pipeline_no_dr = make_modeling_pipeline(classifier=model_candidate, reducer=None)
    scores_no_dr = cross_validate(
        pipeline_no_dr, X, y, cv=cv,
        scoring=['accuracy', 'precision', 'recall', 'f1'],
        return_train_score=False,
    )
    all_scores_no_dr[model_name] = scores_no_dr

rows_no_dr = []
scoring_no_dr = ['accuracy', 'precision', 'recall', 'f1']

for model_name, scores in all_scores_no_dr.items():
    row = {"Model": model_name}
    for metric in scoring_no_dr:
        row[f"{metric}_mean"] = np.mean(scores[f"test_{metric}"])
        row[f"{metric}_std"] = np.std(scores[f"test_{metric}"])
    rows_no_dr.append(row)

summary_df_no_dr = (
    pd.DataFrame(rows_no_dr)
    .sort_values("precision_mean", ascending=False)
    .set_index("Model")
)

float_cols = summary_df_no_dr.select_dtypes(include="float").columns.tolist()
mean_cols = [c for c in float_cols if c.endswith("_mean")]

display(
    summary_df_no_dr.style
    .format("{:.4f}", subset=float_cols)
    .background_gradient(cmap="RdYlGn", axis=0, subset=mean_cols)
)


### 7.b Summary Table - No DR (Paper-formatted)

In [ ]:
le = LabelEncoder().fit(df['class'])
label_order = le.classes_.tolist()
n_labels = len(label_order)

def resolve_ckd_indices(label_order: list[str]) -> tuple[int, int]:
    ckd_idx, notckd_idx = None, None

    for idx, cname in enumerate(label_order):
        lname = cname.lower()
        if 'not' in lname:
            notckd_idx = idx
        elif 'ckd' in lname:
            ckd_idx = idx

    n = len(label_order)

    if ckd_idx is None and notckd_idx is None:
        ckd_idx, notckd_idx = 0, (1 if n > 1 else 0)
    elif ckd_idx is None:
        ckd_idx = next(i for i in range(n) if i != notckd_idx)
    elif notckd_idx is None:
        notckd_idx = next(i for i in range(n) if i != ckd_idx)

    return ckd_idx, notckd_idx

def safe_f1(per_class_f1: np.ndarray, idx: int) -> float:
    return per_class_f1[idx] if idx < len(per_class_f1) else np.nan

ckd_idx, notckd_idx = resolve_ckd_indices(label_order)

rows_for_paper = []

for model_name, model_candidate in models.items():
    pipeline = make_modeling_pipeline(classifier=model_candidate, reducer=None)
    y_pred_oof = cross_val_predict(pipeline, X, y, cv=cv)

    _, _, per_class_f1, _ = precision_recall_fscore_support(
        y, y_pred_oof,
        labels=list(range(n_labels)),
        zero_division=0
    )

    rows_for_paper.append({
        'Model':    model_name,
        'Accuracy': accuracy_score(y, y_pred_oof),
        'Macro-F1': f1_score(y, y_pred_oof, average='macro'),
        'CKD F1':   safe_f1(per_class_f1, ckd_idx),
        'NOTCKD F1':safe_f1(per_class_f1, notckd_idx),
    })

paper_df = (
    pd.DataFrame(rows_for_paper)
      .set_index('Model')
      [['Accuracy', 'Macro-F1', 'CKD F1', 'NOTCKD F1']]
      .sort_values('Macro-F1', ascending=False)
)

display(paper_df.style.format("{:.4f}").background_gradient(cmap="RdYlGn", axis=0))

### 7.c F1 Boxplot - Distribution per Model (No DR)

In [ ]:


ordered_models_no_dr = summary_df_no_dr.index.tolist()

f1_data_no_dr = {
    model_name: scores["test_f1"].tolist()
    for model_name, scores in ((name, all_scores_no_dr[name]) for name in ordered_models_no_dr)
}

fig, ax = plt.subplots(figsize=(12, 6))
pd.DataFrame(f1_data_no_dr).boxplot(ax=ax)
ax.set_title('Macro F1-Distribution per Model\n(No Balancing, No DR, MinMax Scaler, k=5 CV)')
ax.set_ylabel('F1-score')
ax.set_ylim(0.845, 1.02)
ax.tick_params(axis='x', rotation=15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "model_comparison_no_dr_boxplot.png", dpi=300, bbox_inches='tight')
plt.show()


### 7.d ROC Curves - No DR

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for model_name, model_candidate in models.items():
    pipeline_no_dr = make_modeling_pipeline(classifier=model_candidate, reducer=None)
    y_proba = cross_val_predict(pipeline_no_dr, X, y, cv=cv, method="predict_proba")
    fpr, tpr, _ = roc_curve(y, y_proba[:, 1])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves per Model \n (No Balancing, No DR, MinMax Scaler, k=5 CV)")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "model_comparison_no_dr_roc_curve.png", dpi=300, bbox_inches='tight')
plt.show()


## 8. Confusion Matrices (Aggregated over CV Folds)

Out-of-fold predictions via `cross_val_predict` — equivalent to aggregating predictions from all 5 folds, giving a single confusion matrix per model over the full dataset.

In [ ]:
n_models = len(models)
ncols = 3
nrows = int(np.ceil(n_models / ncols))

target_names = df["class"].unique().tolist()  # original class labels in encoded order

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for ax, (model_name, model_candidate) in zip(axes, models.items()):
    pipeline = make_pipeline(model_candidate)
    y_pred   = cross_val_predict(pipeline, X, y, cv=cv)
    cm       = confusion_matrix(y, y_pred)
    disp     = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(model_name)

for ax in axes[n_models:]:
    ax.set_visible(False)

plt.suptitle("Confusion Matrices — PCA 90% var, No Balancing", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. ROC Curves

Out-of-fold probability predictions via `cross_val_predict(method='predict_proba')`. AUC is computed over the aggregated out-of-fold predictions (one curve per model).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for model_name, model_candidate in models.items():
    pipeline    = make_pipeline(model_candidate)
    y_proba     = cross_val_predict(pipeline, X, y, cv=cv, method="predict_proba")
    fpr, tpr, _ = roc_curve(y, y_proba[:, 1])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — PCA 90% var, No Balancing")
ax.legend(loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "spot_checking_model_comparison_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Ranking

In [ ]:
# Main criterion: highest mean F1 across folds
# Tie-breaking criterion: lower std (more consistent across folds)
ranking_df = (
    summary_df[["f1_mean", "f1_std", "accuracy_mean", "precision_mean", "recall_mean"]]
    .sort_values(["f1_mean", "f1_std"], ascending=[False, True])
    .reset_index()
)

ranking_df.index      = ranking_df.index + 1
ranking_df.index.name = "Rank"

float_cols = ranking_df.select_dtypes(include="float").columns.tolist()
ranking_style = (
    ranking_df.style
    .format("{:.4f}", subset=float_cols)
    .background_gradient(cmap="RdYlGn", axis=0, subset=float_cols)
)
display(ranking_style)

save_dataframe_as_figure(
    ranking_df,
    FIGURES_DIR / "spot_checking_model_comparison_ranking.png",
)